In [ ]:
!pip install pandas numpy matplotlib seaborn ipywidgets

In [ ]:
import subprocess
import re
import time
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
# Garantir que a lista comece limpa ao rodar a célula
if 'dados_coletados' not in globals():
    dados_coletados = []

def escanear_wifi(setor_hospitalar):
    mapeamento = {
        "Sala de TV (Recepção Triagem)": "Recepção Triagem",
        "Sala de Jantar (Triagem)": "Triagem",
        "Cozinha (Recepção Abertura Ficha)": "Recepção Abertura Ficha",
        "Quarto 1 (Consultório)": "Consultório",
        "Quarto 2 (Observação)": "Observação",
        "Quarto 3 (RX)": "RX"
    }

    try:
        resultado = subprocess.check_output(
            ["netsh", "wlan", "show", "networks", "mode=bssid"],
            universal_newlines=True,
            encoding='cp850',
            errors='replace',
            timeout=10
        )
    except Exception as e:
        print(f"❌ Erro ao executar netsh: {e}")
        return []

    leituras = []
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    ssid_atual = "Desconhecido"
    bssid_atual = None

    for linha in resultado.split('\n'):
        # ✅ Trabalha com a linha ORIGINAL (sem strip) para detectar indentação
        linha_stripped = linha.strip()
        if not linha_stripped:
            continue

        # SSID começa SEM indentação: "SSID 1 : DORIS_5G"
        if re.match(r'^SSID\s+\d+\s*:', linha_stripped):
            ssid_atual = linha_stripped.split(':', 1)[1].strip()
            bssid_atual = None  # reseta para nova rede

        # BSSID tem 1 nível de indentação: "    BSSID 1 : aa:bb:..."
        elif re.match(r'^BSSID\s+\d+\s*:', linha_stripped):
            # Pega tudo após o PRIMEIRO ':' para não cortar o endereço MAC
            resto = linha_stripped.split(':', 1)[1].strip()
            # O MAC começa depois dos espaços: "92:aa:c3:07:20:f0"
            bssid_atual = resto

        # Sinal: "    Sinal        : 100%"
        elif re.match(r'^Sinal\s*:', linha_stripped, re.IGNORECASE) or \
             re.match(r'^Signal\s*:', linha_stripped, re.IGNORECASE):
            if bssid_atual:
                sinal_str = linha_stripped.split(':', 1)[1].strip().replace('%', '').strip()
                try:
                    sinal = int(sinal_str)
                    leituras.append({
                        "Timestamp": timestamp,
                        "Setor_Original": setor_hospitalar,
                        "Setor_Hospitalar": mapeamento.get(setor_hospitalar, "Desconhecido"),
                        "SSID": ssid_atual,
                        "BSSID": bssid_atual,
                        "RSSI_Percentual": sinal
                    })
                except ValueError:
                    continue

    return leituras

# --- Interface Gráfica Atualizada ---
opcoes_setores = [
    "Sala de TV (Recepção Triagem)",
    "Sala de Jantar (Triagem)",
    "Cozinha (Recepção Abertura Ficha)",
    "Quarto 1 (Consultório)",
    "Quarto 2 (Observação)",
    "Quarto 3 (RX)"
]

dropdown_setor = widgets.Dropdown(
    options=opcoes_setores,
    description='Ambiente:',
)

botao_coletar = widgets.Button(
    description='Coletar Leitura',
    button_style='success', 
    icon='wifi'
)

botao_limpar = widgets.Button(
    description='Limpar Histórico',
    button_style='danger',
    icon='trash'
)

output_area = widgets.Output()

def ao_clicar_coletar(b):
    global dados_coletados
    with output_area:
        clear_output(wait=True)
        setor_selecionado = dropdown_setor.value
        
        # Mapeamento rápido apenas para o texto do print
        nomes_hospital = {
            "Sala de TV (Recepção Triagem)": "Recepção Triagem",
            "Sala de Jantar (Triagem)": "Triagem",
            "Cozinha (Recepção Abertura Ficha)": "Recepção Abertura Ficha",
            "Quarto 1 (Consultório)": "Consultório",
            "Quarto 2 (Observação)": "Observação",
            "Quarto 3 (RX)": "RX"
        }
        
        print(f"📡 Escaneando Wi-Fi para o setor: {nomes_hospital.get(setor_selecionado)}...")
        
        # 3 varreduras seguidas
        for rodada in range(3):
            novas_leituras = escanear_wifi(setor_selecionado)
            if novas_leituras:
                dados_coletados.extend(novas_leituras)
            time.sleep(0.3) 
            
        df_atual = pd.DataFrame(dados_coletados)
        print(f"✅ Executado! Total acumulado na memória: {len(df_atual)} linhas.")
        
        if not df_atual.empty:
            display(df_atual.tail(5))
        else:
            print("⚠️ Nenhuma linha foi adicionada. Verifique se o Wi-Fi não desconectou.")

def ao_clicar_limpar(b):
    global dados_coletados
    dados_coletados = []
    with output_area:
        clear_output()
        print("🗑️ Histórico esvaziado!")

botao_coletar.on_click(ao_clicar_coletar)
botao_limpar.on_click(ao_clicar_limpar)

layout_botoes = widgets.HBox([botao_coletar, botao_limpar])
display(widgets.VBox([dropdown_setor, layout_botoes, output_area]))

In [ ]:
# Gerar o DataFrame consolidado
df_hospital = pd.DataFrame(dados_coletados)

# Salvar em CSV para usar no treinamento da IA / Hackathon
df_hospital.to_csv("dataset_wifi_hospital.csv", index=False)
print("Dataset salvo com sucesso!")

In [ ]:
# 1. Abre o arquivo CSV 
df_verificacao = pd.read_csv("dataset_wifi_hospital.csv")

# 2. Exibe o tamanho do dataset (linhas, colunas)
print(f"Dimensões do Dataset: {df_verificacao.shape[0]} linhas e {df_verificacao.shape[1]} colunas.\n")

# 3. Mostra as primeiras 10 linhas para você conferir a estrutura
display(df_verificacao.head(10))

In [ ]:
# 1. Carrega o arquivo de coletas bruto
df_bruto = pd.read_csv("dataset_wifi_hospital.csv")

# 2. Transforma BSSIDs em colunas (Cada linha vira uma amostra única de tempo e lugar)
df_pivotado = df_bruto.pivot_table(
    index=["Timestamp", "Setor_Hospitalar"], 
    columns="BSSID", 
    values="RSSI_Percentual",
    aggfunc="mean" # Tira a média se houver duplicata exata no mesmo segundo
)

# 3. Trata as redes que não apareceram na leitura:
# Se o notebook não detectou o sinal de um vizinho naquele segundo, o sinal é zero.
df_pivotado = df_pivotado.fillna(0).reset_index()

# 4. Exibe como ficou a estrutura pronta para a IA
print(f"📊 Dataset Estruturado: {df_pivotado.shape[0]} amostras de tempo e {df_pivotado.shape[1]} colunas (características).")
display(df_pivotado.head())

# 5. Salva o dataset final pronto para o treinamento do modelo
df_pivotado.to_csv("dataset_wifi_pronto_para_ia.csv", index=False)
print("💾 Arquivo 'dataset_wifi_pronto_para_ia.csv' gerado com sucesso!")

In [ ]:
# 1. Carrega o arquivo original
df_bruto = pd.read_csv("dataset_wifi_hospital.csv")

# 2. Pivota a tabela (BSSID vira coluna)
df_pivotado = df_bruto.pivot_table(
    index=["Timestamp", "Setor_Hospitalar"], 
    columns="BSSID", 
    values="RSSI_Percentual",
    aggfunc="mean"
).fillna(0).reset_index()

# 3. ANONIMIZAÇÃO: Transforma os BSSIDs reais em "Feature_Antena_X"
colunas_bssid = [col for col in df_pivotado.columns if col not in ["Timestamp", "Setor_Hospitalar"]]
mapeamento_anonimo = {bssid: f"Antena_{i+1:02d}" for i, bssid in enumerate(colunas_bssid)}

# Renomeia as colunas do DataFrame
df_github = df_pivotado.rename(columns=mapeamento_anonimo)

# 4. Remove a coluna de Timestamp (opcional, mas bom para a IA focar apenas no sinal)
df_github = df_github.drop(columns=["Timestamp"])

# 5. Salva o arquivo seguro
df_github.to_csv("dataset_wifi_hospital_clean.csv", index=False)

print("🔒 Dataset anonimizado e estruturado com sucesso!")
print(f"Agora você tem {df_github.shape[1] - 1} colunas de antenas genéricas prontas para a IA.")
display(df_github.head())